In [ ]:
code = 'SBS'
pickle_path = 'C:/PICKLE/'
parameter_path = f'Parameter_{code}.csv'
meta_data_path = f"Parameter_{code}_MetaData.csv"
output_csv_path = f'{code}_output\\'

import sys
sys.path.append(pickle_path)
from BtParameters import *
from BacktestOptions import *

try:
    parameter, parameter_len = get_parameter_data(code, parameter_path)
    meta_data = pd.read_csv(meta_data_path)
    meta_row_nos = get_meta_row_nos(code, meta_data)
    dte_file = get_dte_file(pickle_path)
    os.makedirs(output_csv_path, exist_ok=True)
except Exception as e:
    input(str(e))

In [ ]:
def SBS(bt, from_date, index, code, dte, start_time, end_time, method, sell_sl, sell_trail, sell_sl_trail, sell_track_original, buy_flag, buy_sl, buy_trail, buy_sl_trail, buy_track_original, sell2_flag, om):
    try:
        start_dt = datetime.datetime.combine(from_date, start_time)
        end_dt = datetime.datetime.combine(from_date, end_time)

        ce_scrip, pe_scrip, ce_price, pe_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om=om)
        if ce_scrip is None: return None

        from_candle_close = True if method == 'CC' else False
        entry_time = start_dt

        #### CE ####
        ce_trades = []

        if sell_sl_trail:
            ce_sl_price, ce_sl_flag, _, ce_sl_time, ce_pnl = bt.sl_check_single_leg_with_sl_trail(start_dt, end_dt, ce_scrip, trail=sell_trail, sl_trail=sell_sl_trail, sl=sell_sl, orderside='SELL', from_candle_close=from_candle_close)
        else:
            ce_sl_price, ce_sl_flag, _, _, ce_sl_time, ce_pnl = bt.sl_check_single_leg(start_dt, end_dt, ce_scrip, sl=sell_sl, orderside='SELL', from_candle_close=from_candle_close)

        ce_trades.extend([start_dt, ce_scrip, ce_price, ce_sl_price, ce_sl_flag, ce_sl_time, ce_pnl])

        if sell_track_original:
            _, _, _, _, ce_sl_time, _ = bt.sl_check_single_leg(start_dt, end_dt, ce_scrip, sl=sell_sl, orderside='SELL', from_candle_close=from_candle_close)
            
        #### PE ####
        pe_trades = []

        if sell_sl_trail:
            pe_sl_price, pe_sl_flag, _, pe_sl_time, pe_pnl = bt.sl_check_single_leg_with_sl_trail(start_dt, end_dt, pe_scrip, trail=sell_trail, sl_trail=sell_sl_trail, sl=sell_sl, orderside='SELL', from_candle_close=from_candle_close)
        else:
            pe_sl_price, pe_sl_flag, _, _, pe_sl_time, pe_pnl = bt.sl_check_single_leg(start_dt, end_dt, pe_scrip, sl=sell_sl, orderside='SELL', from_candle_close=from_candle_close)

        pe_trades.extend([start_dt, pe_scrip, pe_price, pe_sl_price, pe_sl_flag, pe_sl_time, pe_pnl])

        if sell_track_original:
            _, _, _, _, pe_sl_time, _ = bt.sl_check_single_leg(start_dt, end_dt, pe_scrip, sl=sell_sl, orderside='SELL', from_candle_close=from_candle_close)
            
        ### CE Buy and Sell ###
        total_ce_pnl = ce_pnl
        if ce_sl_time and buy_flag:

            start_dt = ce_sl_time
            ce_scrip, ce_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om=om, only='CE')
            
            if ce_scrip is None:
                ce_trades.extend(['', '', '', '', False, '', 0])
                ce_trades.extend(['', '', '', '', False, '', 0])
            else:
                # Buy
                if buy_sl_trail:
                    ce_sl_price, ce_sl_flag, _, ce_sl_time, ce_pnl = bt.sl_check_single_leg_with_sl_trail(start_dt, end_dt, ce_scrip, trail=buy_trail, sl_trail=buy_sl_trail, sl=buy_sl, orderside='BUY', from_candle_close=from_candle_close)
                else:
                    ce_sl_price, ce_sl_flag, _, _, ce_sl_time, ce_pnl = bt.sl_check_single_leg(start_dt, end_dt, ce_scrip, sl=buy_sl, orderside='BUY', from_candle_close=from_candle_close)

                ce_trades.extend([start_dt, ce_scrip, ce_price, ce_sl_price, ce_sl_flag, ce_sl_time, ce_pnl])
                total_ce_pnl += ce_pnl

                if buy_track_original:
                    _, _, _, _, ce_sl_time, _ = bt.sl_check_single_leg(start_dt, end_dt, ce_scrip, sl=buy_sl, orderside='BUY', from_candle_close=from_candle_close)

                if ce_sl_time and sell2_flag:

                    start_dt = ce_sl_time
                    ce_scrip, ce_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om=om, only='CE')

                    if ce_scrip is None:
                        ce_trades.extend(['', '', '', '', False, '', 0])
                    else:
                        if sell_sl_trail:
                            ce_sl_price, ce_sl_flag, _, ce_sl_time, ce_pnl = bt.sl_check_single_leg_with_sl_trail(start_dt, end_dt, ce_scrip, trail=sell_trail, sl_trail=sell_sl_trail, sl=sell_sl, orderside='SELL', from_candle_close=from_candle_close)
                        else:
                            ce_sl_price, ce_sl_flag, _, _, ce_sl_time, ce_pnl = bt.sl_check_single_leg(start_dt, end_dt, ce_scrip, sl=sell_sl, orderside='SELL', from_candle_close=from_candle_close)

                        ce_trades.extend([start_dt, ce_scrip, ce_price, ce_sl_price, ce_sl_flag, ce_sl_time, ce_pnl])
                        total_ce_pnl += ce_pnl
                else:
                    ce_trades.extend(['', '', '', '', False, '', 0])
        else:
            ce_trades.extend(['', '', '', '', False, '', 0])
            ce_trades.extend(['', '', '', '', False, '', 0])

        ### PE Buy and Sell ###
        total_pe_pnl = pe_pnl
        if pe_sl_time and buy_flag:

            start_dt = pe_sl_time
            pe_scrip, pe_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om=om, only='PE')
            
            if pe_scrip is None:
                pe_trades.extend(['', '', '', '', False, '', 0])
                pe_trades.extend(['', '', '', '', False, '', 0])
            else:
                # Buy
                if buy_sl_trail:
                    pe_sl_price, pe_sl_flag, _, pe_sl_time, pe_pnl = bt.sl_check_single_leg_with_sl_trail(start_dt, end_dt, pe_scrip, trail=buy_trail, sl_trail=buy_sl_trail, sl=buy_sl, orderside='BUY', from_candle_close=from_candle_close)
                else:
                    pe_sl_price, pe_sl_flag, _, _, pe_sl_time, pe_pnl = bt.sl_check_single_leg(start_dt, end_dt, pe_scrip, sl=buy_sl, orderside='BUY', from_candle_close=from_candle_close)

                pe_trades.extend([start_dt, pe_scrip, pe_price, pe_sl_price, pe_sl_flag, pe_sl_time, pe_pnl])
                total_pe_pnl += pe_pnl

                if buy_track_original:
                    _, _, _, _, pe_sl_time, _ = bt.sl_check_single_leg(start_dt, end_dt, pe_scrip, sl=buy_sl, orderside='BUY', from_candle_close=from_candle_close)

                if pe_sl_time and sell2_flag:

                    start_dt = pe_sl_time
                    pe_scrip, pe_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om=om, only='PE')
                    
                    if pe_scrip is None:
                        pe_trades.extend(['', '', '', '', False, '', 0])
                    else:
                        if sell_sl_trail:
                            pe_sl_price, pe_sl_flag, _, pe_sl_time, pe_pnl = bt.sl_check_single_leg_with_sl_trail(start_dt, end_dt, pe_scrip, trail=sell_trail, sl_trail=sell_sl_trail, sl=sell_sl, orderside='SELL', from_candle_close=from_candle_close)
                        else:
                            pe_sl_price, pe_sl_flag, _, _, pe_sl_time, pe_pnl = bt.sl_check_single_leg(start_dt, end_dt, pe_scrip, sl=sell_sl, orderside='SELL', from_candle_close=from_candle_close)

                        pe_trades.extend([start_dt, pe_scrip, pe_price, pe_sl_price, pe_sl_flag, pe_sl_time, pe_pnl])
                        total_pe_pnl += pe_pnl
                else:
                    pe_trades.extend(['', '', '', '', False, '', 0])
        else:
            pe_trades.extend(['', '', '', '', False, '', 0])
            pe_trades.extend(['', '', '', '', False, '', 0])

        total_pnl = total_ce_pnl + total_pe_pnl
        return [index, code, start_time, end_time, method, sell_sl, sell_trail, sell_sl_trail, sell_track_original, buy_flag, buy_sl, buy_trail, buy_sl_trail, buy_track_original, sell2_flag, om, from_date.date(), from_date.day_name(), dte, entry_time.time(), future_price] + ce_trades + pe_trades
    except Exception as e:
        print(e, [from_date, index, code, dte, start_time, end_time, method, sell_sl, sell_trail, sell_sl_trail, sell_track_original, buy_flag, buy_sl, buy_trail, buy_sl_trail, buy_track_original, sell2_flag, om])
        return

In [ ]:
for row_idx in range(len(meta_data)):

    if row_idx in meta_row_nos and meta_data.loc[row_idx, 'run']:
        try:
            meta_row = meta_data.iloc[row_idx]
            index, dte, from_date, to_date, start_time, end_time, date_lists = get_meta_row_data(meta_row, dte_file)

            log_cols = ('P_Index/P_Strategy/P_StartTime/P_EndTime/P_Method/P_SellSL/P_SellTrail/P_SellSLTrial/P_SellTrackOriginal/P_BuyFlag/P_BuySL/P_BuyTrail/P_BuySLTrial/P_BuyTrackOriginal/P_Sell2Flag/P_OM/Date/Day/DTE/EntryTime/Future')

            ce_log, pe_log = '', ''
            for scrip in ['Sell1', 'Buy', 'Sell2']:
                ce_log += f'/CE.{scrip}.Time/CE.{scrip}.Strike/CE.{scrip}.Price/CE.{scrip}.SL.Price/CE.{scrip}.SL.Flag/CE.{scrip}.SL.Time/CE.{scrip}.PNL'
                pe_log += f'/PE.{scrip}.Time/PE.{scrip}.Strike/PE.{scrip}.Price/PE.{scrip}.SL.Price/PE.{scrip}.SL.Flag/PE.{scrip}.SL.Time/PE.{scrip}.PNL'
            log_cols = (log_cols + ce_log + pe_log).split('/')
            
            for from_date in date_lists:

                file_name = f"{index} {from_date.date()} {code}"
                if not is_file_exists(output_csv_path, file_name, parameter_len):

                    t1 = datetime.datetime.now()
                    print(f"Row-{row_idx} | File-{file_name} | Total-{parameter_len}")

                    bt = IntradayBacktest(pickle_path, index, from_date, start_time, end_time)

                    for idx, i in enumerate(range(0, parameter_len, chunk_size), start=1):
                        chunck_file_name = f"{output_csv_path}{file_name} No-{idx}.parquet"
                        print(chunck_file_name)

                        chunk_parameter = parameter.iloc[i:i+chunk_size]
                        chunk = [SBS(bt,from_date, index, code, dte, row['entry_time'], row['exit_time'], row['method'], row['sell_sl'], row['sell_trail'], row['sell_sl_trail'], row['sell_track_original'], row['buy_flag'], row['buy_sl'], row['buy_trail'], row['buy_sl_trail'], row['buy_track_original'], row['sell2_flag'], row['om']) for idx, row in tqdm(chunk_parameter.iterrows(), total=len(chunk_parameter), colour='GREEN')]
                        save_chunk_data(chunk, log_cols, chunck_file_name)

                    del bt
                    t2 = datetime.datetime.now()
                    print(t2-t1)
        except Exception as e:
            input(str(e))